In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)
from jax_sn.operators.transport_operator import TransportOperatorBatched, TransportOperatorMultipleAnglesVmap, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorMultipleAngles
from jax_sn.domain.geometric_domain import Domain, TopologicalSortedAngle
from jax_sn.domain.cross_sections import CrossSectionData
from jax_sn.solution_domain.basix_fem import BasixLagrangianSimplex
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.solution_domain.solution_domain import SolutionDomain
from jax_sn.operators.moment_operator import MomentOperator, DiscreteOperator
from jax_sn.operators.scatter_operator import GatherScatterOperatorBroadcasted
from jax_sn.operators.multi_group_operator import MultiGroupOperatorScan
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver
from jax_sn.tools.memory_analysis import MultiGroupOperatorSigmaAllParametrised, _filter_jit_f_to_memory, OperatorAnalysis, pytree_nbytes, MultiGroupOperatorEverythingParametrised

from jax_sn.domain.cross_sections import CrossSectionsIndexed
import equinox as eqx
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import lineax as lx
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb
from functions_for_testing import solve_and_plot

In [2]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
import lineax as lx

In [3]:
from finite_elements.test_multigroup_derivative   import modify_setup_mgop,get_rhs, _get_bb_verts_conn_xs_sources, base_region_sizes, base_resolutions, setup_from_verts, _setup_operators
def modify_setup_mgop(dim, degree, modify_verts_xs):
    verts, conn, cross_sections, sources = _get_bb_verts_conn_xs_sources(dim, base_region_sizes(), base_resolutions(), 5.0, 10)
    verts, cross_sections = modify_verts_xs(verts, cross_sections)

    solution_domain, sources, quadrature_set = setup_from_verts(verts, conn, cross_sections, sources, degree, 3)
    
    return _setup_operators(solution_domain, sources, quadrature_set)    

dim, degree = 2,2
bicgstab_solver = lx.BiCGStab(1e-12, 1e-12, max_steps = 160)
lower_triangular_solver = MultiGroupLowerTriangularBlockSolver(bicgstab_solver)
MGOP, source_eg0, _ = modify_setup_mgop(dim, degree, lambda v,c : (v,c))
rhs = get_rhs(MGOP, source_eg0)

W0115 15:28:06.491461   59841 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0115 15:28:06.494541   59795 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [4]:
@eqx.filter_jit
def solve_and_sum(MGOP, rhs, solver):
    solution = lx.linear_solve(MGOP, rhs, solver)
    return jnp.sum(solution.value)



@eqx.filter_jit
def solve_result_everything(theta, MGOP, rhs_base, ltri_solver):    
    MGOP_new = MultiGroupOperatorEverythingParametrised.reconstruct_operator(MGOP, theta)
    solution_value =  lx.linear_solve(MGOP_new, rhs_base, solver=ltri_solver).value
    return jnp.sum(solution_value)

theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)
grad_theta_everything = eqx.filter_jit(jax.grad(solve_result_everything, argnums=0))

In [5]:
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import MultiGroupOperatorScanCheckpoint
MGOP_cp = MultiGroupOperatorScanCheckpoint(MGOP.moment_operator, MGOP.discrete_operator, MGOP.transport_operator, MGOP.mass_operator, MGOP.gather_scatter_operator, MGOP.n_energy_groups, MGOP.transpose_bool)

In [6]:

theta_everything_base.vertices.device

CudaDevice(id=0)

In [14]:
from jax_sn.tools.memory_analysis import custom_timeit
def print_memory_parametrisation(theta, MGOP, attribute_name, attribute_method):
    
    
    if attribute_name is None:
        op = MGOP
    else:
        op = getattr(MGOP, attribute_name)

    rhs = jnp.ones_like(op.in_structure())

    @eqx.filter_jit
    def sum_attr(mgop, theta, rhs, attribute_name):
        mgop1 = MultiGroupOperatorEverythingParametrised.reconstruct_operator(mgop, theta)
        if attribute_name is None:
            op = mgop1
        else:
            op = getattr(mgop1, attribute_name)

    
        solution = getattr(op, attribute_method)(rhs)
        
        return jnp.sum(solution)
    
    f_grad = eqx.filter_jit(jax.value_and_grad(sum_attr, argnums=1))
    
    print(f"{attribute_name}.{attribute_method} memory usage:\n\t Direct {_filter_jit_f_to_memory(sum_attr, MGOP, theta, rhs, attribute_name)} MB", f"\n\t Grad {_filter_jit_f_to_memory(f_grad, MGOP, theta, rhs, attribute_name)} MB")

    def sum_attr_p():
        return sum_attr(MGOP, theta, rhs, attribute_name)
    
    def cu():
        return f_grad(MGOP, theta, rhs, attribute_name)
    
    print("  Timing:\n\t","Direct", custom_timeit(sum_attr_p, number = 3, ignore_n=1)[0], "\n\t", "Grad", custom_timeit(cu, number = 3, ignore_n=1)[0])


print("No checkpointing:")
print("==============================")
print_memory_parametrisation(theta_everything_base, MGOP, None, "mv")
print_memory_parametrisation(theta_everything_base, MGOP, "moment_operator", "mv")
print_memory_parametrisation(theta_everything_base, MGOP, "transport_operator", "inv")
print_memory_parametrisation(theta_everything_base, MGOP, "transport_operator", "mv")
print_memory_parametrisation(theta_everything_base, MGOP, "mass_operator", "mv")
print("==============================\n\n")
print("With checkpointing:")
print("==============================")

print_memory_parametrisation(theta_everything_base, MGOP_cp, None, "mv")
print_memory_parametrisation(theta_everything_base, MGOP_cp, "moment_operator", "mv")
print_memory_parametrisation(theta_everything_base, MGOP_cp, "transport_operator", "inv")
print_memory_parametrisation(theta_everything_base, MGOP_cp, "transport_operator", "mv")
print_memory_parametrisation(theta_everything_base, MGOP_cp, "mass_operator", "mv")
print("==============================")

No checkpointing:
None.mv memory usage:
	 Direct {'peak': 14.699943542480469, 'argument': 3.95343017578125, 'output': 3.0517578125e-05, 'temp': 10.785820007324219} MB 
	 Grad {'peak': 37.21913242340088, 'argument': 3.95343017578125, 'output': 0.01052093505859375, 'temp': 33.27348327636719} MB
  Timing:
	 Direct 0.1029207706451416 
	 Grad 0.2264847755432129
moment_operator.mv memory usage:
	 Direct {'peak': 4.159461975097656, 'argument': 1.667724609375, 'output': 3.0517578125e-05, 'temp': 2.4919586181640625} MB 
	 Grad {'peak': 4.169952392578125, 'argument': 1.667724609375, 'output': 0.01052093505859375, 'temp': 2.4919815063476562} MB
  Timing:
	 Direct 0.0006968975067138672 
	 Grad 0.001093745231628418
transport_operator.inv memory usage:
	 Direct {'peak': 8.068130493164062, 'argument': 2.265655517578125, 'output': 3.0517578125e-05, 'temp': 5.807014465332031} MB 
	 Grad {'peak': 11.856826782226562, 'argument': 2.265655517578125, 'output': 0.01052093505859375, 'temp': 9.594161987304688}

In [8]:
@eqx.filter_jit
def solve_fwd(mgop, rhs, solver):
    return lx.linear_solve(mgop, rhs, solver=solver).value


@eqx.filter_jit
def solve_lower_tri(mgop, rhs, solver):
    ltri_solver=  MultiGroupLowerTriangularBlockSolver(solver)
    return lx.linear_solve(mgop, rhs, solver=ltri_solver).value

@eqx.filter_jit
def solve_bwd(mgop, rhs, solver):    
    return lx.linear_solve(mgop.transpose(), rhs, solver=solver).value

@eqx.filter_jit
def solve_bwd_lower_tri(mgop, rhs, solver):    
    ltri_solver=  MultiGroupLowerTriangularBlockSolver(solver)
    return lx.linear_solve(mgop.transpose(), rhs, solver=ltri_solver).value

def print_costs_solves(MGOP, rhs_base, bicgstab_solver):
    sgop = MGOP.to_single_group_operator(0)
    rhs_base = jnp.ones_like(MGOP.in_structure())
    bicgstab_solver = lx.BiCGStab(1e-12, 1e-12, max_steps = 160)
    print("Transport Operator")
    print(OperatorAnalysis.analyze_memory(sgop.transport_operator, "inv"))
    print("Single group operator")
    print(OperatorAnalysis.analyze_memory(sgop, "mv"))
    print("Multi group operator")
    print(OperatorAnalysis.analyze_memory(MGOP, "mv"))
    print("Forward solve")
    print(_filter_jit_f_to_memory(solve_fwd, MGOP, rhs_base, bicgstab_solver ))
    print("Forward tri solve")
    print(_filter_jit_f_to_memory(solve_lower_tri, MGOP, rhs_base, bicgstab_solver ))

    print("Backward solve")
    print(_filter_jit_f_to_memory(solve_bwd, MGOP, rhs_base, bicgstab_solver ))
    print("Backward tri solve")
    print(_filter_jit_f_to_memory(solve_bwd_lower_tri, MGOP, rhs_base, bicgstab_solver ))
print("No checkpointing:")
print("==============================")
print_costs_solves(MGOP, rhs, bicgstab_solver)
print("==============================\n\n")
print("With checkpointing:")
print("==============================")
print_costs_solves(MGOP_cp, rhs, bicgstab_solver)
print("==============================")

No checkpointing:
Transport Operator
{'peak': 8.083381652832031, 'argument': 2.34576416015625, 'output': 1.6479721069335938, 'temp': 4.092658996582031}
Single group operator
{'peak': 9.702384948730469, 'argument': 1.575103759765625, 'output': 0.8239974975585938, 'temp': 7.306770324707031}
Multi group operator
{'peak': 14.728691101074219, 'argument': 4.047027587890625, 'output': 3.2959213256835938, 'temp': 7.423774719238281}
Forward solve
{'peak': 34.557419776916504, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 29.676513671875}
Forward tri solve
{'peak': 20.68700408935547, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 14.850837707519531}
Backward solve
{'peak': 41.425599098205566, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 34.04541015625}
Backward tri solve
{'peak': 26.049964904785156, 'argument': 4.0999908447265625, 'output': 3.2959213256835938, 'temp': 19.38623809814453}


With checkpointing:
Transport Operato

In [9]:
ltri_solver = MultiGroupLowerTriangularBlockSolver(bicgstab_solver)

In [10]:
from jax_sn.tools.memory_analysis import OperatorThetaParametrised
from jax_sn.solution_domain import PrecomputedGeometricData
from jax_sn.domain.cross_sections import CrossSectionData
from jax_sn.domain.geometric_domain import compute_face_normal_vectors_from_3d_simplices
from jax_sn.solution_domain.basix_fem import _precompute_jacobians_simplex

theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)

@eqx.filter_jit
def solve_result_everything(theta, MGOP, rhs_base, ltri_solver):    
    MGOP_new = MultiGroupOperatorEverythingParametrised.reconstruct_operator(MGOP, theta)
    solution_value =  lx.linear_solve(MGOP_new, rhs_base, solver=ltri_solver).value
    return jnp.sum(solution_value)

grad_theta_everything = eqx.filter_jit(jax.grad(solve_result_everything, argnums=0))


def print_mem_timing(theta, mgop, rhs_base, ltri_solver):
    @eqx.filter_jit
    def f(theta):
        mgop_new = MultiGroupOperatorEverythingParametrised.reconstruct_operator(mgop, theta)
        solution_value =  lx.linear_solve(mgop_new, rhs_base, solver=ltri_solver).value
        return jnp.sum(solution_value)
    
    f_grad = eqx.filter_jit(jax.value_and_grad(f, argnums=0))
    
    print(f"Solve memory usage:\n\t Direct {_filter_jit_f_to_memory(f, theta)} MB", f"\n\t Grad {_filter_jit_f_to_memory(f_grad, theta)} MB")

    def sum_attr_p():
        return f(theta)
    
    def cu():
        return f_grad(theta)
    
    print("  Timing:\n\t","Direct", custom_timeit(sum_attr_p, number = 1, ignore_n=0)[0], "\n\t", "Grad", custom_timeit(cu, number = 1, ignore_n=0)[0])

In [11]:
lx.linear_solve(MGOP, rhs, solver=ltri_solver).stats

{'num_steps_per_group': Array([21, 59, 50, 31], dtype=int64)}

16.1

In [ ]:
print("Everything parametrised")
theta_everything_base = MultiGroupOperatorEverythingParametrised.theta_base(MGOP)

print("Without checkpointing Lower Triangular")
print("======================================")
print_mem_timing(theta_everything_base, MGOP, rhs, ltri_solver)
print("======================================\n\n")
print("With checkpointing Lower Triangular")
print("======================================")
print_mem_timing(theta_everything_base, MGOP_cp, rhs, ltri_solver)
print("======================================")


Everything parametrised
Without checkpointing Lower Triangular
Solve memory usage:
	 Direct {'peak': 21.331924438476562, 'argument': 0.010467529296875, 'output': 3.0517578125e-05, 'temp': 17.879043579101562} MB 
	 Grad {'peak': 38.641395568847656, 'argument': 0.010467529296875, 'output': 0.01052093505859375, 'temp': 32.04462432861328} MB
  Timing:
	 Direct 7.9548468589782715 
	 Grad 17.658540964126587


With checkpointing Lower Triangular
Solve memory usage:
	 Direct {'peak': 21.331924438476562, 'argument': 0.010467529296875, 'output': 3.0517578125e-05, 'temp': 17.879043579101562} MB 
	 Grad {'peak': 34.684242248535156, 'argument': 0.010467529296875, 'output': 0.01052093505859375, 'temp': 29.049522399902344} MB
  Timing:
	 Direct 7.968562364578247 
	 Grad 17.6671359539032
